# GAMPFN vs TabICLv2 on quarter-hourly electricity-load prediction

This notebook adapts `TSFM/notebooks/tabpfn_load.ipynb` to compare the stage-1 GAMPFN checkpoint with the released TabICLv2 regressor. Both models receive exactly the same chronological train/test split and features for each of the 96 quarter-hour instants.

The stage-1 model was trained on sequences of length 1,024. Test predictions are therefore made in fixed chunks so that roughly 730 context rows plus each query chunk remain within that regime.

In [ ]:
from datetime import datetime
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tabicl import TabICLRegressor

REPO_ROOT = Path("/Users/andrei/Desktop/GAMPFN")
DATA_PATH = Path("/Users/andrei/Desktop/TSFM/data_preprocessed_final_enhanced_2.csv")
GAMPFN_CHECKPOINT = REPO_ROOT / "checkpoints/gam/stage1/step-80000.ckpt"
TABICLV2_CHECKPOINT = REPO_ROOT / "checkpoints/baselines/tabicl-regressor-v2-20260212.ckpt"

BEGIN_TRAIN = datetime(2021, 7, 1)
BEGIN_TEST = datetime(2023, 7, 1)
END_TEST = datetime(2024, 7, 1)
N_ESTIMATORS = 2  # Use 8 for the final comparison.
TEST_CHUNK_SIZE = 256
DEVICE = None  # TabICL chooses CUDA, then XPU, MPS, or CPU.
MAX_INSTANTS = None  # Set to a small integer for a smoke run.

for path in (DATA_PATH, GAMPFN_CHECKPOINT, TABICLV2_CHECKPOINT):
    if not path.is_file():
        raise FileNotFoundError(path)

## Load the same prediction task

Only the requested time window is retained in memory. Preprocessing remains inside each `TabICLRegressor`, so nothing is fitted on the test targets or test period.

In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=["Date_local"])
data = data[(data["Date_local"] >= BEGIN_TRAIN) & (data["Date_local"] < END_TEST)].copy()
data["JourFerie"] = data["JourFerie"].astype(int)

feature_columns = [column for column in data.columns if column not in {"Consommation", "Date_local"}]
instants = [(hour, minute) for hour in range(24) for minute in (0, 15, 30, 45)]
if MAX_INSTANTS is not None:
    instants = instants[:MAX_INSTANTS]

print(f"Rows in selected period: {len(data):,}")
print(f"Features: {len(feature_columns)}")
print(f"Instants to evaluate: {len(instants)}")
print(f"Columns containing missing values: {data[feature_columns].isna().any().sum()}")

## Load each checkpoint once

Calling `fit` prepares a new in-context dataset; it does not update model weights. The small subclass avoids re-reading the same checkpoint 96 times. Representation caching makes chunked prediction practical.

In [ ]:
class ReusableTabICLRegressor(TabICLRegressor):
    def _load_model(self):
        if not hasattr(self, "model_"):
            super()._load_model()


def predict_in_chunks(model, X, chunk_size):
    chunks = [model.predict(X.iloc[start : start + chunk_size]) for start in range(0, len(X), chunk_size)]
    return np.concatenate(chunks)


common_model_options = {
    "allow_auto_download": False,
    "n_estimators": N_ESTIMATORS,
    "random_state": 0,
    "device": DEVICE,
    "kv_cache": "repr",
}
models = {
    "GAMPFN": ReusableTabICLRegressor(model_path=GAMPFN_CHECKPOINT, **common_model_options),
    "TabICLv2": ReusableTabICLRegressor(model_path=TABICLV2_CHECKPOINT, **common_model_options),
}

## Run the paired 96-instant comparison

Each quarter-hour instant is treated as a separate in-context regression problem, matching the reference notebook.

In [ ]:
prediction_rows = []
instant_metric_rows = []

for instant_index, (hour, minute) in enumerate(instants, start=1):
    is_instant = (data["Date_local"].dt.hour == hour) & (data["Date_local"].dt.minute == minute)
    train = data[
        is_instant & (data["Date_local"] >= BEGIN_TRAIN) & (data["Date_local"] < BEGIN_TEST)
    ].reset_index(drop=True)
    test = data[
        is_instant & (data["Date_local"] >= BEGIN_TEST) & (data["Date_local"] < END_TEST)
    ].reset_index(drop=True)

    X_train = train[feature_columns]
    y_train = train["Consommation"].to_numpy()
    X_test = test[feature_columns]
    y_test = test["Consommation"].to_numpy()
    test_dates = test["Date_local"].to_numpy()

    print(
        f"[{instant_index:02d}/{len(instants)}] {hour:02d}:{minute:02d} "
        f"train={len(train)} test={len(test)}"
    )

    for model_name, model in models.items():
        started = time.perf_counter()
        model.fit(X_train, y_train)
        prediction = predict_in_chunks(model, X_test, TEST_CHUNK_SIZE)
        elapsed = time.perf_counter() - started

        rmse = np.sqrt(mean_squared_error(y_test, prediction))
        mae = mean_absolute_error(y_test, prediction)
        instant_metric_rows.append(
            {
                "Model": model_name,
                "Hour": hour,
                "Minute": minute,
                "Train_Size": len(train),
                "Test_Size": len(test),
                "RMSE": rmse,
                "MAE": mae,
                "Seconds": elapsed,
            }
        )
        prediction_rows.extend(
            {
                "Date_local": date,
                "Hour": hour,
                "Minute": minute,
                "Model": model_name,
                "Actual": actual,
                "Predicted": predicted,
            }
            for date, actual, predicted in zip(test_dates, y_test, prediction)
        )
        print(f"    {model_name:<9} RMSE={rmse:8.2f} MW MAE={mae:8.2f} MW time={elapsed:6.1f}s")

predictions_df = pd.DataFrame(prediction_rows).sort_values(["Date_local", "Model"]).reset_index(drop=True)
instant_metrics_df = pd.DataFrame(instant_metric_rows).sort_values(["Hour", "Minute", "Model"]).reset_index(drop=True)
predictions_df["Error"] = predictions_df["Predicted"] - predictions_df["Actual"]
predictions_df["Absolute_Error"] = predictions_df["Error"].abs()

## Overall paired results

In [ ]:
overall_rows = []
for model_name, model_results in predictions_df.groupby("Model"):
    actual = model_results["Actual"].to_numpy()
    predicted = model_results["Predicted"].to_numpy()
    overall_rows.append(
        {
            "Model": model_name,
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "MAE": mean_absolute_error(actual, predicted),
            "MAPE (%)": 100 * np.mean(np.abs((actual - predicted) / actual)),
            "R2": r2_score(actual, predicted),
        }
    )

overall_metrics = pd.DataFrame(overall_rows).set_index("Model").sort_index()
tabicl_rmse = overall_metrics.loc["TabICLv2", "RMSE"]
overall_metrics["RMSE improvement vs TabICLv2 (%)"] = 100 * (tabicl_rmse - overall_metrics["RMSE"]) / tabicl_rmse
display(overall_metrics.style.format(precision=3))

## Per-instant comparison

In [ ]:
instant_comparison = instant_metrics_df.pivot(index=["Hour", "Minute"], columns="Model", values="RMSE")
instant_comparison["GAMPFN improvement (%)"] = (
    100 * (instant_comparison["TabICLv2"] - instant_comparison["GAMPFN"]) / instant_comparison["TabICLv2"]
)
instant_comparison = instant_comparison.sort_values("GAMPFN improvement (%)", ascending=False)

print(f"GAMPFN wins on {(instant_comparison['GAMPFN improvement (%)'] > 0).mean():.1%} of instants.")
print("Best GAMPFN instants:")
display(instant_comparison.head(10).style.format(precision=2))
print("Worst GAMPFN instants:")
display(instant_comparison.tail(10).style.format(precision=2))

## Visual comparison

In [ ]:
week_start = pd.Timestamp("2024-02-01")
week_end = week_start + pd.Timedelta(days=7)
week = predictions_df[
    (predictions_df["Date_local"] >= week_start) & (predictions_df["Date_local"] < week_end)
].copy()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

actual_week = week.drop_duplicates("Date_local").sort_values("Date_local")
axes[0, 0].plot(actual_week["Date_local"], actual_week["Actual"], color="black", label="Actual", linewidth=2)
for model_name, model_results in week.groupby("Model"):
    model_results = model_results.sort_values("Date_local")
    axes[0, 0].plot(model_results["Date_local"], model_results["Predicted"], label=model_name, alpha=0.8)
axes[0, 0].set_title("One-week load prediction")
axes[0, 0].set_ylabel("Consumption (MW)")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

for model_name, model_results in predictions_df.groupby("Model"):
    axes[0, 1].scatter(model_results["Actual"], model_results["Predicted"], s=5, alpha=0.15, label=model_name)
limits = [predictions_df[["Actual", "Predicted"]].min().min(), predictions_df[["Actual", "Predicted"]].max().max()]
axes[0, 1].plot(limits, limits, "k--", linewidth=1)
axes[0, 1].set_title("Predicted versus actual")
axes[0, 1].set_xlabel("Actual (MW)")
axes[0, 1].set_ylabel("Predicted (MW)")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

for model_name, model_results in predictions_df.groupby("Model"):
    axes[1, 0].hist(model_results["Error"], bins=100, alpha=0.45, label=model_name)
axes[1, 0].axvline(0, color="black", linestyle="--")
axes[1, 0].set_title("Prediction-error distribution")
axes[1, 0].set_xlabel("Prediction error (MW)")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

plot_comparison = instant_comparison.sort_index().reset_index()
slot = 4 * plot_comparison["Hour"] + plot_comparison["Minute"] // 15
axes[1, 1].bar(slot, plot_comparison["GAMPFN improvement (%)"], width=0.8)
axes[1, 1].axhline(0, color="black", linestyle="--")
axes[1, 1].set_title("GAMPFN RMSE improvement by quarter-hour")
axes[1, 1].set_xlabel("Quarter-hour of day")
axes[1, 1].set_ylabel("Improvement over TabICLv2 (%)")
axes[1, 1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Optional export

Results remain in memory by default. Set the flag below only when you want to create CSV files.

In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    output_dir = REPO_ROOT / "results"
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions_df.to_csv(output_dir / "gampfn_vs_tabiclv2_load_predictions.csv", index=False)
    instant_metrics_df.to_csv(output_dir / "gampfn_vs_tabiclv2_load_by_instant.csv", index=False)
    overall_metrics.to_csv(output_dir / "gampfn_vs_tabiclv2_load_overall.csv")
    print(f"Saved results to {output_dir}")